In [1]:
import os
os.makedirs('tests', exist_ok=True)

In [4]:
%%writefile tests/test_predict.py

import pickle
import pytest
import numpy as np

with open('models/lin_reg.bin', 'rb') as f:
    dv, model = pickle.load(f)

def predict_duration(trip):
    features = {
        'PULocationID': str(trip['PULocationID']),
        'DOLocationID': str(trip['DOLocationID']),
        'trip_distance': trip['trip_distance']
    }
    X = dv.transform([features])
    return float(model.predict(X)[0])

# Test 1: output is a float
def test_predict_returns_float():
    trip = {'PULocationID': 65, 'DOLocationID': 233, 'trip_distance': 6.2}
    result = predict_duration(trip)
    assert isinstance(result, float)

# Test 2: output is a positive number
def test_predict_positive_duration():
    trip = {'PULocationID': 65, 'DOLocationID': 233, 'trip_distance': 6.2}
    result = predict_duration(trip)
    assert result > 0

# Test 3: different locations give different durations
def test_different_locations_different_duration():
    trip_a = {'PULocationID': 65,  'DOLocationID': 233, 'trip_distance': 6.2}
    trip_b = {'PULocationID': 100, 'DOLocationID': 50,  'trip_distance': 6.2}
    assert predict_duration(trip_a) != predict_duration(trip_b)

# Test 4: duration is within realistic range
def test_predict_realistic_range():
    trip = {'PULocationID': 65, 'DOLocationID': 233, 'trip_distance': 6.2}
    result = predict_duration(trip)
    assert 1 <= result <= 120

Overwriting tests/test_predict.py


In [5]:
!pytest tests/test_predict.py -v

============================= test session starts ==============================
platform linux -- Python 3.12.1, pytest-9.0.2, pluggy-1.6.0 -- /usr/local/py-utils/venvs/pytest/bin/python
cachedir: .pytest_cache
rootdir: /workspaces/nyc-taxi-trip-analysis
plugins: anyio-4.12.1, Faker-40.18.0
collected 4 items                                                              

tests/test_predict.py::test_predict_returns_float PASSED                 [ 25%]
tests/test_predict.py::test_predict_positive_duration PASSED             [ 50%]
tests/test_predict.py::test_different_locations_different_duration PASSED [ 75%]
tests/test_predict.py::test_predict_realistic_range PASSED               [100%]

============================== 4 passed in 2.39s ===============================


In [6]:
!git add .
!git commit -m "Add monitoring, tests and CI/CD workflow"
!git push origin main

[main 45e00f6] Add monitoring, tests and CI/CD workflow
 32 files changed, 9594 insertions(+)
 create mode 100644 Dockerfile
 create mode 100644 best_practices.ipynb
 create mode 100644 data/green_tripdata_2026-01.parquet
 create mode 100644 data/green_tripdata_2026-02.parquet
 create mode 100644 data/updated_df.parquet
 create mode 100644 data_exploration.ipynb
 create mode 100644 deployment.ipynb
 create mode 100644 docker-compose.yml
 create mode 100644 mlflow.db
 create mode 100644 mlflow_tracking.ipynb
 create mode 100644 mlruns/1/models/m-5f33eb1a684f4931924f776b4790b36f/artifacts/MLmodel
 create mode 100644 mlruns/1/models/m-5f33eb1a684f4931924f776b4790b36f/artifacts/conda.yaml
 create mode 100644 mlruns/1/models/m-5f33eb1a684f4931924f776b4790b36f/artifacts/model.pkl
 create mode 100644 mlruns/1/models/m-5f33eb1a684f4931924f776b4790b36f/artifacts/python_env.yaml
 create mode 100644 mlruns/1/models/m-5f33eb1a684f4931924f776b4790b36f/artifacts/requirements.txt
 create mode 100644 

In [7]:
!cat .github/workflows/ci.yml

cat: .github/workflows/ci.yml: No such file or directory


In [8]:
import os
os.makedirs('.github/workflows', exist_ok=True)
print("Folders created!")

Folders created!


In [9]:
%%writefile .github/workflows/ci.yml

name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.12'

    - name: Install dependencies
      run: |
        pip install scikit-learn numpy pytest flask

    - name: Run tests
      run: |
        pytest tests/test_predict.py -v

Writing .github/workflows/ci.yml


In [10]:
!cat .github/workflows/ci.yml


name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.12'

    - name: Install dependencies
      run: |
        pip install scikit-learn numpy pytest flask

    - name: Run tests
      run: |
        pytest tests/test_predict.py -v


In [11]:
!git add .github/workflows/ci.yml
!git commit -m "Add CI workflow"
!git push origin main

[main f527076] Add CI workflow
 1 file changed, 28 insertions(+)
 create mode 100644 .github/workflows/ci.yml
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 622 bytes | 622.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/NikitaJagtap39/nyc-taxi-trip-analysis
   45e00f6..f527076  main -> main
